## 5.2.2数据分组与聚合

1. transform 
- 是 groupby 操作的一个非常有用的功能。transform 允许你对分组后的数据应用一个函数，然后将结果转换回与原始数据相同的形状，但仅包含应用函数的列。 

2. Apply和transform主要区别：
  - 结果形状：apply 的结果可以是任意形状，而 transform 的结果必须与原始数据集的形状相同。
  - 用途：apply 用于更复杂的数据转换，可能改变数据的大小；transform 用于保持数据形状不变的情况，通常用于创建新的列。
  - 速度：由于 transform 操作是针对每个分组独立进行的，通常比 apply 更快，尤其是在大数据集上。

  - 根据你的具体需求选择合适的方法。
    - 如果你需要对分组数据进行汇总统计并保留分组结构，使用 transform；
    - 如果你需要根据分组数据得到一个不同的数据结构，使用 apply。


In [1]:
import numpy as np
import pandas as pd
import random

In [2]:
random.seed(0)
np.random.seed(0)
df = pd.DataFrame({
    "工号": np.arange(801, 809),
    "部门": np.random.choice(['A', 'B'], size=8),
    "年龄": np.random.randint(15, 50, 8),
    "绩效": np.random.randint(5, 50, 8),
})
df

,工号,部门,年龄,绩效
0,801,A,24,6
1,802,B,34,43
2,803,B,36,44
3,804,A,38,28
4,805,B,21,29
5,806,B,39,22
6,807,B,39,42
7,808,B,27,30


### 练习：新增平均年龄列，填充各部门的平均年龄

#### 分组

In [3]:
df['平均年龄'] = df.groupby('部门')['年龄'].transform('mean')  
df

,工号,部门,年龄,绩效,平均年龄
0,801,A,24,6,31.000000
1,802,B,34,43,32.666667
2,803,B,36,44,32.666667
3,804,A,38,28,31.000000
4,805,B,21,29,32.666667
5,806,B,39,22,32.666667
6,807,B,39,42,32.666667
7,808,B,27,30,32.666667


In [4]:
df.groupby("部门").count()

,工号,年龄,绩效,平均年龄
部门,,,,
A,2,2,2,2
B,6,6,6,6


In [5]:
# 按照部门显示数据
for dept, grp in df.groupby(["部门"]):
    print(f"部门{dept[0]}")
    display(grp)

部门A


,工号,部门,年龄,绩效,平均年龄
0,801,A,24,6,31.0
3,804,A,38,28,31.0


部门B


,工号,部门,年龄,绩效,平均年龄
1,802,B,34,43,32.666667
2,803,B,36,44,32.666667
4,805,B,21,29,32.666667
5,806,B,39,22,32.666667
6,807,B,39,42,32.666667
7,808,B,27,30,32.666667


In [6]:
df.groupby(["部门"]).agg('mean') 
# 按部门分组以后求均值（或求和）的过程，称为聚合（agg）。
df.groupby(["部门"]).mean()
# 聚合的简写方式，分组后直接求mean，这里显然工号列没有求mean的意义

,工号,年龄,绩效,平均年龄
部门,,,,
A,802.500000,31.000000,17.0,31.000000
B,805.166667,32.666667,35.0,32.666667


In [7]:
df.groupby("部门",as_index=False)[['年龄', '绩效']].mean()
# as_index=False：该参数控制分组后的结果是否以分组列作为索引。
# 设置为 False 时，分组列会作为结果数据框的普通列，而不是索引。

,部门,年龄,绩效
0,A,31.000000,17.0
1,B,32.666667,35.0


In [8]:
df.groupby("部门")[['年龄', '绩效']].mean().reset_index()

,部门,年龄,绩效
0,A,31.000000,17.0
1,B,32.666667,35.0


In [9]:
df.groupby('部门').绩效.mean()

部门
A    17.0
B    35.0
Name: 绩效, dtype: float64

#### 两级列表

In [10]:
df3= df.groupby('部门').agg(
  {'绩效': ['mean', 'sum', 'std'],
   '年龄':'median'})
df3

绩效                     年龄
    mean  sum        std median
部门                             
A   17.0   34  15.556349   31.0
B   35.0  210   9.208692   35.0

In [11]:
df3.columns# 两级列名

MultiIndex([('绩效',   'mean'),
            ('绩效',    'sum'),
            ('绩效',    'std'),
            ('年龄', 'median')],
           )

In [12]:
df3[[('绩效','mean'), ('绩效', 'sum')]] #列名分级别 要想访问mean这列 必须加上上一级别绩效

绩效     
    mean  sum
部门           
A   17.0   34
B   35.0  210

In [13]:
df3.绩效

,mean,sum,std
部门,,,
A,17.0,34,15.556349
B,35.0,210,9.208692


In [14]:
df3.绩效['std'] #访问二级列数据

部门
A    15.556349
B     9.208692
Name: std, dtype: float64

### groupby与apply

In [15]:
df.groupby('部门')['绩效'].mean()# series
df.groupby("部门")[['绩效']].apply('mean')# dataframe
#这里apply聚合后 数据的结构不同 只有两行

,绩效
部门,
A,17.0
B,35.0


### apply与lambda

In [16]:
df2 = df.copy()
df2['新绩效'] = df2['绩效'].apply(lambda x: x * 2)  
#匿名函数lambda,这里针对每个绩效列的x，计算x乘以2
df2

,工号,部门,年龄,绩效,平均年龄,新绩效
0,801,A,24,6,31.000000,12
1,802,B,34,43,32.666667,86
2,803,B,36,44,32.666667,88
3,804,A,38,28,31.000000,56
4,805,B,21,29,32.666667,58
5,806,B,39,22,32.666667,44
6,807,B,39,42,32.666667,84
7,808,B,27,30,32.666667,60


### groupby与transform数据转换

In [17]:
df

,工号,部门,年龄,绩效,平均年龄
0,801,A,24,6,31.000000
1,802,B,34,43,32.666667
2,803,B,36,44,32.666667
3,804,A,38,28,31.000000
4,805,B,21,29,32.666667
5,806,B,39,22,32.666667
6,807,B,39,42,32.666667
7,808,B,27,30,32.666667


In [18]:
df.groupby('部门')['绩效'].mean()
#df.groupby("部门")[['绩效']].apply(np.mean) #这里apply聚合后 数据的结构不同 只有两行

部门
A    17.0
B    35.0
Name: 绩效, dtype: float64

In [19]:
df4 = df.copy()
df4['部门平均绩效'] = df.groupby('部门')['绩效'].transform('mean')  
#使用 transform 方法，根据聚合后的结果填充，保持与原始DataFrame相同的行数。这样，每个部门的每个记录都会有相应的平均绩效值。
df4

,工号,部门,年龄,绩效,平均年龄,部门平均绩效
0,801,A,24,6,31.000000,17.0
1,802,B,34,43,32.666667,35.0
2,803,B,36,44,32.666667,35.0
3,804,A,38,28,31.000000,17.0
4,805,B,21,29,32.666667,35.0
5,806,B,39,22,32.666667,35.0
6,807,B,39,42,32.666667,35.0
7,808,B,27,30,32.666667,35.0


In [20]:
df4 = df.copy()
df4['部门平均绩效'] = df.groupby('部门')['绩效'].transform(lambda x: x.mean())  
df4


,工号,部门,年龄,绩效,平均年龄,部门平均绩效
0,801,A,24,6,31.000000,17.0
1,802,B,34,43,32.666667,35.0
2,803,B,36,44,32.666667,35.0
3,804,A,38,28,31.000000,17.0
4,805,B,21,29,32.666667,35.0
5,806,B,39,22,32.666667,35.0
6,807,B,39,42,32.666667,35.0
7,808,B,27,30,32.666667,35.0


In [21]:
df4 = df.copy()
df4['部门平均绩效'] = df.groupby('部门')['绩效'].transform('mean')
df4['部门内绩效差异'] = df4.绩效 - df4.部门平均绩效
df4

,工号,部门,年龄,绩效,平均年龄,部门平均绩效,部门内绩效差异
0,801,A,24,6,31.000000,17.0,-11.0
1,802,B,34,43,32.666667,35.0,8.0
2,803,B,36,44,32.666667,35.0,9.0
3,804,A,38,28,31.000000,17.0,11.0
4,805,B,21,29,32.666667,35.0,-6.0
5,806,B,39,22,32.666667,35.0,-13.0
6,807,B,39,42,32.666667,35.0,7.0
7,808,B,27,30,32.666667,35.0,-5.0


In [22]:
df5 = df.copy()
df5['新增平均年龄'] = df.groupby('部门')['年龄'].transform(lambda x: x.mean()+5)  # 计算每个部门的平均年龄，并将结果添加到原始数据中
df5

,工号,部门,年龄,绩效,平均年龄,新增平均年龄
0,801,A,24,6,31.000000,36.000000
1,802,B,34,43,32.666667,37.666667
2,803,B,36,44,32.666667,37.666667
3,804,A,38,28,31.000000,36.000000
4,805,B,21,29,32.666667,37.666667
5,806,B,39,22,32.666667,37.666667
6,807,B,39,42,32.666667,37.666667
7,808,B,27,30,32.666667,37.666667



## 排名函数或秩函数[`rank()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rank.html)

In [23]:
pd.Series([30, 25, 10, 25, 25, 40]).rank()

0    5.0
1    3.0
2    1.0
3    3.0
4    3.0
5    6.0
dtype: float64

In [24]:
df.groupby('部门').绩效.rank()

0    1.0
1    5.0
2    6.0
3    2.0
4    2.0
5    1.0
6    4.0
7    3.0
Name: 绩效, dtype: float64

In [25]:
df.groupby('部门').绩效.rank(ascending=False)

0    2.0
1    2.0
2    1.0
3    1.0
4    5.0
5    6.0
6    3.0
7    4.0
Name: 绩效, dtype: float64

In [26]:
df.groupby('部门')['绩效'].transform("rank", ascending=False)

0    2.0
1    2.0
2    1.0
3    1.0
4    5.0
5    6.0
6    3.0
7    4.0
Name: 绩效, dtype: float64

In [27]:
df4['部门内绩效排名'] = df.groupby('部门').绩效.rank(ascending=False)
df4

,工号,部门,年龄,绩效,平均年龄,部门平均绩效,部门内绩效差异,部门内绩效排名
0,801,A,24,6,31.000000,17.0,-11.0,2.0
1,802,B,34,43,32.666667,35.0,8.0,2.0
2,803,B,36,44,32.666667,35.0,9.0,1.0
3,804,A,38,28,31.000000,17.0,11.0,1.0
4,805,B,21,29,32.666667,35.0,-6.0,5.0
5,806,B,39,22,32.666667,35.0,-13.0,6.0
6,807,B,39,42,32.666667,35.0,7.0,3.0
7,808,B,27,30,32.666667,35.0,-5.0,4.0
